# Optional experiment: the same queue in your browser

Use this instead of the downloadable practical if you prefer. Each code cell includes its own setup, so it does not depend on having run an earlier cell. You only need to change the lines marked CHANGE FIRST. Saved output gives you a reading route if execution is unavailable. No Jupyter installation is required.

## A. Check a fixed trace

Predict the remaining queue for arrivals `[1,0,1,1,0]` and services `[0,1,0,1,1]`. Run the first cell. Then change both lists to three minutes: arrivals `[1,1,1]`, services `[0,1,0]`. Expected new queue: `[1,1,2]`.

In [1]:
def queue_trace(arrivals, services):
    if len(arrivals) != len(services) or any(x not in (0, 1) for x in arrivals + services):
        raise ValueError("equal-length lists of binary events required")
    queue, served, history = 0, 0, []
    for arrival, service in zip(arrivals, services):
        queue += arrival
        if queue and service:
            queue -= 1
            served += 1
        history.append(queue)
    return {"history": history, "arrivals": sum(arrivals), "served": served}



# CHANGE FIRST: the two event lists below.
result = queue_trace([1, 0, 1, 1, 0], [0, 1, 0, 1, 1])
print("Remaining queue:", result["history"])
print("Arrivals:", result["arrivals"], "Served:", result["served"])


Remaining queue: [1, 0, 1, 1, 0]
Arrivals: 3 Served: 3


## B. Add probabilities

The function uses exactly the lab signature: `simulate(minutes, arrival_probability, service_probability, seed)`. It returns history, arrivals, and served counts. Start with the supplied call; run twice, then change only seed 7 to 8. Explain what repeats and why. The helper definitions are supplied so you can focus on the experiment.

In [2]:
from random import Random
from statistics import mean

def queue_trace(arrivals, services):
    if len(arrivals) != len(services) or any(x not in (0, 1) for x in arrivals + services):
        raise ValueError("equal-length lists of binary events required")
    queue, served, history = 0, 0, []
    for arrival, service in zip(arrivals, services):
        queue += arrival
        if queue and service:
            queue -= 1
            served += 1
        history.append(queue)
    return {"history": history, "arrivals": sum(arrivals), "served": served}


def simulate(minutes, arrival_probability, service_probability, seed):
    if type(minutes) is not int or minutes < 0:
        raise ValueError("minutes must be a non-negative integer")
    if not (0 <= arrival_probability <= 1 and 0 <= service_probability <= 1):
        raise ValueError("probabilities must lie in [0, 1]")
    rng = Random(seed)
    arrivals, services = [], []
    for _ in range(minutes):
        arrivals.append(int(rng.random() < arrival_probability))
        services.append(int(rng.random() < service_probability))
    return queue_trace(arrivals, services)


def summarise(result):
    history = result["history"]
    return {"final": history[-1] if history else 0,
            "maximum": max(history, default=0),
            "mean_queue": mean(history) if history else None}



# CHANGE FIRST: the final argument is the seed.
result = simulate(10, 0.6, 0.5, 7)
assert result["history"][-1] == result["arrivals"] - result["served"]
print("Remaining queue:", result["history"])
print("Summary:", summarise(result))


Remaining queue: [0, 0, 0, 1, 1, 1, 2, 2, 2, 2]
Summary: {'final': 2, 'maximum': 2, 'mean_queue': 1.1}


## C. Check a fair comparison

Predict what increasing service probability should do. Run the two scenarios below with the same duration, arrivals, and 20 seeds. The output measures people remaining, not minutes waited. Explain one missing real-world factor. The next cell is a worked comparison; try your prediction first.

In [3]:
from random import Random
from statistics import mean

def queue_trace(arrivals, services):
    if len(arrivals) != len(services) or any(x not in (0, 1) for x in arrivals + services):
        raise ValueError("equal-length lists of binary events required")
    queue, served, history = 0, 0, []
    for arrival, service in zip(arrivals, services):
        queue += arrival
        if queue and service:
            queue -= 1
            served += 1
        history.append(queue)
    return {"history": history, "arrivals": sum(arrivals), "served": served}


def simulate(minutes, arrival_probability, service_probability, seed):
    if type(minutes) is not int or minutes < 0:
        raise ValueError("minutes must be a non-negative integer")
    if not (0 <= arrival_probability <= 1 and 0 <= service_probability <= 1):
        raise ValueError("probabilities must lie in [0, 1]")
    rng = Random(seed)
    arrivals, services = [], []
    for _ in range(minutes):
        arrivals.append(int(rng.random() < arrival_probability))
        services.append(int(rng.random() < service_probability))
    return queue_trace(arrivals, services)


def summarise(result):
    history = result["history"]
    return {"final": history[-1] if history else 0,
            "maximum": max(history, default=0),
            "mean_queue": mean(history) if history else None}



for service in (0.5, 0.8):
    means = [summarise(simulate(120, 0.6, service, seed))["mean_queue"] for seed in range(20)]
    print(f"Service {service:.1f}: mean of run means={mean(means):.3f} people; "
          f"range={min(means):.3f} to {max(means):.3f}")


Service 0.5: mean of run means=7.379 people; range=1.333 to 16.200
Service 0.8: mean of run means=0.581 people; range=0.175 to 1.267


## Explain your result

Complete: “I changed ___. I kept ___ fixed. I measured ___. The model omits ___.” Verification checks these rules; validation needs evidence about the real system. An optional waiting-time extension would record individual arrival and service times.

[Next: 3. Check your understanding](https://tutors.dev/quiz/lc-cs/topic-11-modelling-simulation/quiz-01-check) · [4. Worked answers](https://tutors.dev/note/lc-cs/topic-11-modelling-simulation/note-02-feedback).